# QWT-JEPA — huan luyen tren Kaggle

Model: mot anh RGB `256x256` + mot cua so IMU `128x6` -> phuc hoi ca hai.
QWT 2D dual-tree cho anh, Haar 1D cho IMU, hai CNN encoder, shared gated MLP
fusion, hai decoder, va nhanh JEPA (teacher EMA + predictor).

**Settings can dat:** Accelerator = `GPU T4 x2` (hoac P100) | Internet = `ON`
(de `git clone`).

**Add data:** dataset `tartanair-v2-256` (output cua notebook prepare-data).
Tu phien 2 tro di, them ca output cua phien truoc de `--resume`.

Moi phien Kaggle toi da 12h. Quy trinh: chay den khi gan het gio -> Save Version
-> phien sau add output cu lam input roi `--resume`.

In [ ]:
# ---- Cell 0: kiem tra GPU va moi truong -------------------------------------
import subprocess, torch, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| n_gpu', torch.cuda.device_count())
print('python', sys.version.split()[0])

In [ ]:
# ---- Cell 1: lay code --------------------------------------------------------
# Day qwt_jepa_version2 len GitHub truoc (tu may local):
#   cd /home/buidinhkhoi/Thuctap && git add qwt_jepa_version2 && git commit -m 'v2' && git push
import pathlib, shutil, os

REPO = 'https://github.com/leesanghyoek/qwt_jepa.git'
CODE = pathlib.Path('/kaggle/working/code')
if CODE.exists():
    shutil.rmtree(CODE)
!git clone -q $REPO {CODE}

# Package nam o <repo>/qwt_jepa_version2/qjepa
PKG_PARENT = CODE / 'qwt_jepa_version2'
assert (PKG_PARENT / 'qjepa').is_dir(), f'khong thay qjepa trong {PKG_PARENT}'
os.chdir(PKG_PARENT)
sys.path.insert(0, str(PKG_PARENT))
print('cwd =', os.getcwd())

In [ ]:
# ---- Cell 2: tim dataset trong /kaggle/input --------------------------------
# TU DONG tim thay vi hard-code slug: ten dataset co the khac giua cac tai khoan.
import pathlib

def find_data_root():
    for base in sorted(pathlib.Path('/kaggle/input').glob('*')):
        for cand in [base] + sorted(p for p in base.glob('*') if p.is_dir()):
            # Mot data root hop le co <env>/<Data_x>/<Pxxx>/imu/imu_time.{npy,txt}
            hit = list(cand.glob('*/Data_*/P*/imu/imu_time.*'))
            if hit:
                return cand
    raise FileNotFoundError('khong tim thay tartanair-v2 trong /kaggle/input')

DATA_ROOT = find_data_root()
print('DATA_ROOT =', DATA_ROOT)
envs = sorted({p.name for p in DATA_ROOT.glob('*') if p.is_dir()})
print(f'{len(envs)} environment:', envs)

# Checkpoint cua phien truoc (neu da add output cu lam input)
PREV = next((p for p in pathlib.Path('/kaggle/input').glob('*/outputs/main_qwt/last.pt')), None)
print('checkpoint phien truoc:', PREV)

In [ ]:
# ---- Cell 3: audit du lieu (gate G0) ----------------------------------------
# Xac minh tan suat va timestamp tu DU LIEU, khong coi 10/100 Hz la mac dinh.
!python -m qjepa audit-data --root {DATA_ROOT} --window 128 --out /kaggle/working/outputs/audit

In [ ]:
# ---- Cell 4: gate G1 - round-trip va gradient cua transform -----------------
# Phai PASS truoc khi train: neu QWT khong tai tao duoc thi moi so lieu sau do vo nghia.
!python -m qjepa check-transform

In [ ]:
# ---- Cell 5: build manifest + normalization stats ---------------------------
# Ghep anh-IMU theo timestamp, chia split theo trajectory, tinh mu/scale tren
# rieng train clean. Chay mot lan cho moi dataset.
MANIFEST = '/kaggle/working/outputs/manifest'
!python -m qjepa build-manifest --root {DATA_ROOT} --out {MANIFEST} --window 128

In [ ]:
# ---- Cell 6: gate G2 - smoke train 30 step ----------------------------------
!python -m qjepa smoke-train --config configs/kaggle_t4.yaml --manifest {MANIFEST} \
    --steps 30 --output-dir /kaggle/working/outputs/smoke

In [ ]:
# ---- Cell 7: gate G3 - overfit 16 sample co dinh ----------------------------
# Muc tieu: MSE anh, accel va gyro deu giam >= 30% so voi input nhieu.
# Dung config BALANCED: voi imu_weight=1 nhu dac ta, gradient nhanh IMU nho
# hon nhanh anh ~88 lan va nhanh IMU khong hoc. Xem TRAINING_REPORT muc 4.
# Neu gate nay fail thi KHONG chay full train - phai chan doan truoc.
!python -m qjepa overfit --config configs/kaggle_balanced.yaml --manifest {MANIFEST} \
    --samples 16 --steps 800 --output-dir /kaggle/working/outputs/overfit

In [ ]:
# ---- Cell 8: TRAIN -----------------------------------------------------------
# Stage A (2000 step, lambda_J=0) roi Stage B (8000 step, JEPA ramp).
# Phien dau: khong co --resume. Tu phien 2: them --resume PREV.
#
# Uoc luong: ~0.3 s/step tren T4 voi batch 8 fp16 -> 10.000 step ~ 50 phut/1000 step.
# Dat --steps thap hon max_optimizer_steps de dung truoc gioi han 12h, roi Save Version.
import shlex

resume = f'--resume {PREV}' if PREV else ''
STEPS = 10000           # giam xuong (vi du 6000) neu sap het gio phien
cmd = (f'python -m qjepa train --config configs/kaggle_balanced.yaml '
       f'--manifest {MANIFEST} --steps {STEPS} --log-every 50 '
       f'--output-dir /kaggle/working/outputs/main_qwt {resume}')
print(cmd)
!{cmd}

In [ ]:
# ---- Cell 9: danh gia va export ---------------------------------------------
OUT = '/kaggle/working/outputs/main_qwt'
!python -m qjepa evaluate --config configs/kaggle_balanced.yaml --manifest {MANIFEST} \
    --checkpoint {OUT}/last.pt --split valid
!python -m qjepa export --config configs/kaggle_balanced.yaml --manifest {MANIFEST} \
    --checkpoint {OUT}/last.pt

In [ ]:
# ---- Cell 10: don dep truoc khi Save Version --------------------------------
# Gioi han commit cua /kaggle/working la 20GB. Chi giu checkpoint va log.
import pathlib, shutil
shutil.rmtree('/kaggle/working/code', ignore_errors=True)
shutil.rmtree('/kaggle/working/outputs/smoke', ignore_errors=True)
for p in sorted(pathlib.Path('/kaggle/working/outputs').rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size/2**20:8.1f} MB  {p.relative_to("/kaggle/working")}')

## Chay tiep o phien sau

1. Bam **Save Version -> Save & Run All (Commit)**.
2. Tu output do tao **New Dataset** (hoac dung truc tiep output cua notebook).
3. Phien sau: **Add data** them output vua tao. Cell 2 se tu tim `last.pt`, va
   Cell 8 tu them `--resume`.

`--resume` khoi phuc model, optimizer, scheduler, teacher EMA, step, stage va
RNG. No tu choi chay neu `manifest_hash` hoac image transform khac voi
checkpoint — de khong vo tinh ghep hai run khac cau hinh vao mot duong metric.

Dung `--init` thay cho `--resume` khi muon bat dau thi nghiem MOI tu trong so cu.

## Ablation (moi cai la mot run rieng, cung budget/seed/split)

| Config | Doi chung |
| --- | --- |
| `configs/no_jepa.yaml` | dong gop cua JEPA (control: lambda_J=0 toan bo) |
| `configs/no_cross.yaml` | dong gop cua cross-modal fusion |
| `configs/haar_baseline.yaml` | dong gop cua QWT (thay bang Haar 12 kenh) |
| `configs/stage_c.yaml` | nhieu nang hon: bias + drift + spike |